# Assignment 2 — The Bookstore Intelligence Mission
**Business Data and Text Mining | Web Scraping**  
Target: https://books.toscrape.com/

## Setup

In [ ]:
import time
import requests
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE_URL = "https://books.toscrape.com/"
HEADERS = {"User-Agent": "Mozilla/5.0 (student web-scraping assignment)"}

# Rating words -> numbers (the "rating challenge")
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

## Part 1 — Reconnaissance

| Data Needed | HTML Tag / Class Found | Notes |
|---|---|---|
| Book Title | `<h3><a title="...">` | Full title is in the `title` attribute |
| Price | `<p class="price_color">` | Text like `£51.77`; convert to float |
| Rating | `<p class="star-rating Three">` | Second CSS class holds the rating word |
| Availability | `<p class="instock availability">` | Text `In stock`; strip whitespace |
| Product URL (Bonus 2) | `<h3><a href="...">` | Relative link; join with page URL |
| Next page | `<li class="next"><a href="page-2.html">` | Followed until it no longer exists |

## Part 2 — Capture the Data (First Page)

In [ ]:
def parse_books(soup, page_url):
    """Extract every book on one page into a list of dicts."""
    records = []
    for card in soup.find_all("article", class_="product_pod"):
        link = card.h3.a
        title = link["title"]                                   # full title (not truncated)
        price = float(card.find("p", class_="price_color")
                          .text.replace("£", "").replace("Â", "").strip())
        rating_word = card.find("p", class_="star-rating")["class"][1]
        rating = RATING_MAP[rating_word]                        # 'Three' -> 3
        availability = card.find("p", class_="instock availability").text.strip()
        url = urljoin(page_url, link["href"])                   # bonus 2: product URL
        records.append({"Title": title, "Price": price, "Rating": rating,
                        "Availability": availability, "URL": url})
    return records


def get_soup(url):
    """Download a page and return a BeautifulSoup object."""
    response = requests.get(url, headers=HEADERS, timeout=15)
    response.raise_for_status()
    response.encoding = "utf-8"          # fixes the 'Â£' encoding issue
    return BeautifulSoup(response.text, "html.parser")


# Test on the first page only
first_page = get_soup(BASE_URL)
page1 = pd.DataFrame(parse_books(first_page, BASE_URL))
print(f"Books on page 1: {len(page1)}")
page1.head()

## Part 3 — Scale the Operation (Boss Level: All Pages)

In [ ]:
def scrape_catalogue(start_url=BASE_URL, max_pages=None, delay=0.3):
    """Scrape pages by following the 'next' link. max_pages=None -> scrape everything."""
    all_records, url, page_no = [], start_url, 0
    while url:
        soup = get_soup(url)
        all_records.extend(parse_books(soup, url))
        page_no += 1
        print(f"Page {page_no:>2} scraped | total books so far: {len(all_records)}")

        if max_pages and page_no >= max_pages:
            break
        next_btn = soup.find("li", class_="next")
        url = urljoin(url, next_btn.a["href"]) if next_btn else None
        time.sleep(delay)                 # be polite to the server
    return pd.DataFrame(all_records)


# Boss level: all pages. (For a quick test use scrape_catalogue(max_pages=3))
df = scrape_catalogue()

### 3.1 — Check the DataFrame

In [ ]:
print(df.shape)
print(df.dtypes)
print("\nMissing values:\n", df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())
df.head()

### 3.2 — Save the Dataset

In [ ]:
df.to_csv("competitor_books.csv", index=False)
print("Saved competitor_books.csv with", len(df), "rows")

# In Colab, uncomment to download the file:
# from google.colab import files
# files.download("competitor_books.csv")

## Part 4 — Intelligence Analysis

### 4.1 — Questions 1 to 5

In [ ]:
df["In_Stock"] = df["Availability"].str.contains("In stock", case=False)

# Q1
most_exp = df.loc[df["Price"].idxmax()]
print("1. Most expensive book:")
print(f"   {most_exp['Title']}  |  £{most_exp['Price']:.2f}  |  {most_exp['Rating']} stars\n")

# Q2
cheapest = df.loc[df["Price"].idxmin()]
print("2. Cheapest book:")
print(f"   {cheapest['Title']}  |  £{cheapest['Price']:.2f}  |  {cheapest['Rating']} stars\n")

# Q3
print(f"3. Average book price: £{df['Price'].mean():.2f}\n")

# Q4
five_star = df[df["Rating"] == 5]
print(f"4. Books with a 5-star rating: {len(five_star)}\n")

# Q5
pct_stock = df["In_Stock"].mean() * 100
print(f"5. Books in stock: {pct_stock:.1f}%  ({df['In_Stock'].sum()} of {len(df)})\n")

### 4.2 — Questions 6 and 7

In [ ]:
# Q6: ties broken by lowest price
print("6. Top 5 highest-rated books (ties broken by lowest price):")
top5 = df.sort_values(["Rating", "Price"], ascending=[False, True]).head(5)
display(top5[["Title", "Price", "Rating"]].reset_index(drop=True))

# Q7
cheapest_5 = five_star.loc[five_star["Price"].idxmin()]
print("7. Cheapest 5-star book:")
print(f"   {cheapest_5['Title']}  |  £{cheapest_5['Price']:.2f}")

### 4.3 — Bonus 3 and 4: Rating Counts

In [ ]:
rating_counts = df["Rating"].value_counts().sort_index()
display(rating_counts.rename("Number of books").to_frame())

ax = rating_counts.plot(kind="bar", color="steelblue", edgecolor="black", figsize=(7, 4))
ax.set_title("Number of Books by Star Rating")
ax.set_xlabel("Star rating")
ax.set_ylabel("Number of books")
plt.xticks(rotation=0)
for i, v in enumerate(rating_counts):
    ax.text(i, v + 2, str(v), ha="center")
plt.tight_layout()
plt.show()

## Part 5 — The CEO Has £100

In [ ]:
import numpy as np

BUDGET = 100.00

def knapsack_exact(prices, ratings, budget):
    """0/1 knapsack. Returns indices of chosen items (max total rating, then min cost)."""
    w = [int(round(p * 100)) for p in prices]       # pence
    cap = int(round(budget * 100))
    n, NEG = len(w), -10**9
    dp = np.full(cap + 1, NEG, dtype=np.int64)       # dp[c] = best rating spending EXACTLY c
    dp[0] = 0
    keep = np.zeros((n, cap + 1), dtype=bool)
    for i in range(n):
        wi, vi = w[i], ratings[i]
        if wi > cap:
            continue
        cand = dp[:cap + 1 - wi] + vi
        better = cand > dp[wi:]
        keep[i, wi:] = better
        dp[wi:] = np.where(better, cand, dp[wi:])
    best = dp.max()
    c = int(np.where(dp == best)[0][0])              # cheapest cost reaching the best rating
    chosen = []
    for i in range(n - 1, -1, -1):
        if keep[i, c]:
            chosen.append(i)
            c -= w[i]
    return sorted(chosen)

# Only in-stock books are eligible
pool = df[df["In_Stock"]].reset_index(drop=True)
idx = knapsack_exact(pool["Price"].tolist(), pool["Rating"].tolist(), BUDGET)
basket = pool.loc[idx, ["Title", "Price", "Rating"]].sort_values("Price").reset_index(drop=True)

display(basket)
print(f"Books selected      : {len(basket)}")
print(f"Total cost          : £{basket['Price'].sum():.2f}  (budget £{BUDGET:.2f})")
print(f"Total rating score  : {basket['Rating'].sum()}")

## Part 6 — Business Decision

### 6.1 — Supporting Evidence

In [ ]:
avg_price = df["Price"].mean()
avg_5 = five_star["Price"].mean()
share_5 = len(five_star) / len(df) * 100

# Evidence 1: top-rated books priced below the catalogue average and in stock
sweet_spot = df[(df["Rating"] >= 4) & (df["Price"] < avg_price) & (df["In_Stock"])]

# Evidence 2: do higher ratings cost more?
price_by_rating = df.groupby("Rating")["Price"].mean().round(2)
corr = df["Rating"].corr(df["Price"])

print(f"Evidence 1: {len(sweet_spot)} books ({len(sweet_spot)/len(df)*100:.1f}% of catalogue) are rated 4+ stars, "
      f"in stock and cheaper than the £{avg_price:.2f} average.")
print(f"Evidence 2: 5-star books average £{avg_5:.2f} vs £{avg_price:.2f} overall; "
      f"rating vs price correlation is {corr:.2f}.")
print(f"Evidence 3: only {share_5:.1f}% of books are 5-star, so they are scarce and worth securing.")
print("\nAverage price by rating:")
display(price_by_rating.to_frame("Avg price (£)"))

### 6.2 — Recommendation

In [ ]:
from IPython.display import Markdown
direction = "barely changes with" if abs(corr) < 0.2 else ("rises with" if corr > 0 else "falls with")
Markdown(f"""
### Recommendation
We would prioritise the **{len(sweet_spot)} in-stock books rated 4+ stars that sit below the £{avg_price:.2f} average price**
({len(sweet_spot)/len(df)*100:.1f}% of the catalogue). The data shows price {direction} star rating
(correlation {corr:.2f}), so strong ratings do not require a premium price and well-reviewed books can be stocked at competitive prices. With only {share_5:.1f}% of titles rated 5 stars at an average of £{avg_5:.2f}, these scarce top-rated
books can anchor the launch range.
""")